# Practical 1: bytes and struct

Computers need ways to represent data that can travel over a wire. A command, a wavelength such as `600`, or a measurement must become bytes before we can send it between the computer and the plate reader. Both sides need to agree on what those bytes mean.

In this session, we will inspect captured bytes, send commands to the Pico, and turn its replies back into useful values. This practical introduces the Python tools we need to make those conversions and understand what we are sending and receiving.

Run cells with Shift+Enter. Predict each result before running it. Python's standard library is enough.

By the end you should be able to explain the difference between text and bytes, index and slice bytes, and convert between Python values and bytes using the supplied `struct` formats.


In [ ]:
import struct
import math

## 1. Create and inspect bytes
A `bytes` object is an immutable sequence of integers from 0 to 255. Use `bytes.fromhex()` to convert a hex string into bytes. Each pair of hex digits represents one byte; spaces are optional.


In [ ]:
data = bytes.fromhex("02 58 41 ff")
print(data)
print(data.hex(" "))
print(len(data))
print(list(data))


Python prints some bytes as characters: byte `0x58` may appear as `X`. The value has not changed. Use `.hex(" ")` to see every byte in the same notation.

**Exercise 1.** Convert `00 10 80 ff` to bytes using `bytes.fromhex(...)`. Predict its length and the result of `list(...)`, then check them.


In [ ]:
# sample = bytes.fromhex(...)
# print(len(sample), list(sample))


## 2. Index, slice, and concatenate
Indexing returns an integer. Slicing returns a bytes object. Indices start at zero; the slice end is excluded.


In [ ]:
data = bytes.fromhex("02 58 41 ff")
print(data[0], type(data[0]))
print(data[:2], type(data[:2]))
print(data[-1])
print((data[:2] + data[2:]).hex(" "))


**Exercise 2.** Extract the middle two bytes (`58 41`). Append `00` to `data` by concatenating it with `bytes.fromhex("00")`.


In [ ]:
# middle = ...
# extended = ...
# print(middle.hex(" "), extended.hex(" "))


## 3. Text is not bytes
`str` holds text. `.encode("ascii")` converts ASCII text to bytes; `.decode("ascii")` converts those bytes back. The encoding is part of the agreement about how to interpret the bytes.


In [ ]:
text = "READ"
encoded = text.encode("ascii")
print(encoded, encoded.hex(" "))
print(encoded.decode("ascii"))
print(b"02 58".hex(" "))
print(bytes.fromhex("02 58").hex(" "))


**Exercise 3.** Encode `"600"` as ASCII and inspect its length and hex representation. Why is this three bytes? Why is `b"02 58"` five bytes rather than two?


In [ ]:
# text_bytes = ...
# print(len(text_bytes), text_bytes.hex(" "))


## 4. Pack and unpack an integer
`struct.pack(format, values...)` returns bytes. `struct.unpack(format, data)` returns a tuple, even for one value.

Use `>H` for this device's two-byte whole numbers, from 0 to 65535. This device sends the high byte first; keep the `>` in the format. The formats you need are supplied throughout the workshop.


In [ ]:
packed = struct.pack(">H", 600)
print(packed.hex(" "))
print(struct.unpack(">H", packed))
number, = struct.unpack(">H", packed)
print(number)
assert number == 600


**Exercise 4.** Pack 450 with `>H`. Unpack `01 00` with `>H`. Store the result as an integer rather than a one-element tuple. Check a pack/unpack round trip for 0, 255, 256, and 65535.


In [ ]:
# print(struct.pack(..., 450).hex(" "))
# number, = struct.unpack(..., bytes.fromhex("01 00"))
# for value in (0, 255, 256, 65535):
#     assert struct.unpack(..., struct.pack(..., value))[0] == value


## 5. Formats and multiple values
Use these formats in the examples:

| Format | Meaning | Bytes |
|---|---|---|
| `B` | Whole number, 0–255 | 1 |
| `>H` | Whole number, 0–65535 | 2 |
| `>f` | Floating-point number | 4 |

Combine letters after the `>` to pack multiple values. For example, `>BH` packs a one-byte whole number followed by a two-byte whole number. Use `struct.calcsize(...)` to find the total size.


In [ ]:
packed = struct.pack(">BH", 7, 600)
print(packed.hex(" "))
print(struct.calcsize(">BH"))
first, second = struct.unpack(">BH", packed)
print(first, second)


**Exercise 5.** Pack the values 3, 1000, and 20 using `>BHH`. Predict the total size. Unpack them and check that you get the original values back.


In [ ]:
# packed_values = struct.pack(">BHH", ...)
# print(packed_values.hex(" "), struct.calcsize(">BHH"))
# assert struct.unpack(">BHH", packed_values) == (3, 1000, 20)


## 6. Floating-point values
Use `>f` for the four-byte floating-point numbers in this example. Small rounding differences can occur; use `math.isclose()` when comparing them.


In [ ]:
packed_float = struct.pack(">f", 0.153)
value, = struct.unpack(">f", packed_float)
print(packed_float.hex(" "))
print(value)
print(value == 0.153)
assert math.isclose(value, 0.153, abs_tol=1e-7)


**Exercise 6.** Pack and unpack 1.5 using `>f`. Verify that it uses four bytes and that the unpacked value is 1.5.


In [ ]:
# packed = ...
# value, = ...
# assert len(packed) == 4
# assert value == 1.5


### Check your understanding
Without running code, explain:
1. The difference between `data[0]` and `data[:1]`.
2. The difference between `"600".encode("ascii")` and `struct.pack(">H", 600)`.
3. Why `struct.unpack(">H", b"\x02\x58")` returns `(600,)`.
4. How many bytes `>BH` uses, and how many values unpacking it returns.
